# J1 — Du fichier brut au premier modèle : les données, puis la décision

**Séance 1.** On part d'un fichier de rendez-vous et on va jusqu'à une soumission classée — puis on
regarde les données de près, parce qu'un modèle ne vaut pas mieux qu'elles.

> **La règle :** un score sans référence ne veut rien dire, et un constat sans chiffre n'en est pas un.

**Livrable :** une soumission valide · le lien de votre notebook · le tableau d'essais (hypothèse,
changement, attendu, mesuré) · un rapport de **5 constats chiffrés** et **3 hypothèses de features**.

In [ ]:
# les bibliothèques du cours, et les versions pour lesquelles il a été écrit :
#        pandas 3.0.6 · numpy 2.4.3 · scikit-learn 1.9.0 · jinja2
#
# Le code tourne AUSSI sous pandas 2 (c'est ce que Kaggle fournit) : les deux sont testés.
#
# SUR KAGGLE : rien à faire. Les bibliothèques sont déjà installées et l'INTERNET EST COUPÉ :
# une commande d'installation y échoue. Ce n'est pas une erreur de votre part.
#
# EN LOCAL, une seule fois — dans un terminal :
#     pip install "pandas==3.0.6" "numpy==2.4.3" "scikit-learn==1.9.0" "jinja2"
# dans une cellule d'un notebook local, la même chose avec un point d'exclamation devant.

# Ce que vous avez, à cet instant :
import importlib

for nom in ("pandas", "numpy", "sklearn", "jinja2"):
    try:
        print(f"{nom:<8}", importlib.import_module(nom).__version__)
    except ImportError:
        print(f"{nom:<8} absent — voir les lignes d'installation ci-dessus")

In [ ]:
# les outils
import glob
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

In [ ]:
# où sont les fichiers (le notebook marche sur Kaggle ET en local)
# On CHERCHE train.csv au lieu de supposer un chemin. Kaggle a déjà changé ses points de
# montage (/kaggle/input/<identifiant> puis /kaggle/input/competitions/<identifiant>) : un
# chemin recopié à la main se périme, une recherche non.
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie", os.path.join("..", "sortie"),
                            os.path.join("..", "Donnees_competition", "sortie")])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)

### Lire la table, et préparer les deux jeux

Une colonne lue comme un nombre alors que c'est du texte fait apprendre du bruit : le score baisse un
peu, personne ne voit l'erreur, et on cherche le problème dans le modèle pendant deux semaines.
Ici on écarte trois familles de colonnes, sans les justifier aujourd'hui : les **identifiants** (ils
désignent une ligne, ils ne la décrivent pas), les **dates brutes et les mesures saisies en texte**
(leur forme n'est pas exploitable), et une colonne **mise de côté** — on verra pourquoi en regardant
la table.
La cellule construit **la même table** pour l'entraînement et pour le test, mêmes colonnes et mêmes
modalités : c'est la règle des deux côtés.

In [ ]:
# lire les deux tableaux, et leur préparer LA MÊME table
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

# Ce qu'on écarte aujourd'hui, sans l'expliquer : les identifiants, les colonnes connues APRÈS
# l'issue (elles n'existeraient pas au moment de décider), les dates brutes et le texte libre.
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]    # mise de côté : on la retrouvera en explorant

def matrice(cols):
    """Renvoie (X, colonnes catégorielles, colonnes écartées) pour un tableau donné."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()

    # `is_numeric_dtype` et non `dtype == "object"` — pandas 3 a un dtype `str` dédié.
    # Et une colonne à très forte cardinalité n'est PAS catégorielle utilisable : les modèles
    # d'arbres plafonnent à 255 modalités. Au-delà, il faut un autre traitement.
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]

    for c in cat:
        X[c] = X[c].fillna("").astype("category")

    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:                      # MÊMES modalités que l'entraînement, sinon décalage silencieux
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

y = train[CIBLE]

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("la cible \n")
print(f"cible : {100*y.mean():.2f} % de positifs")

print("après préparation \n")
print("colonnes de travail :", X.shape[1])
print("écartées (cardinalité > 255) :", TROP_FIN)
print("colonnes catégorielles :", CAT)

In [ ]:
# À vous — la table de travail
# Regardez ce que la préparation a produit : combien de colonnes, lesquelles sont
# catégorielles, combien de lignes de chaque côté.

print("colonnes de travail :", X.shape[1])
print("catégorielles :", CAT)
print()
# TODO : écartez une colonne de plus, de votre choix, et regardez ce que ça change.
#        Notez la ligne dans le tableau d'essais (section 5).

### La référence — ce que vaut un modèle qui ne fait rien

Avant de mesurer un modèle, il faut savoir ce que donne un modèle qui répond toujours la même chose.
Ici, répondre « présent » à tout le monde vaut **81,75 % d'exactitude**, sans une seule ligne
d'apprentissage : la cible est déséquilibrée.
C'est pour cette raison qu'un score seul ne veut rien dire — 82 % peut être excellent ou nul, selon ce
que le déséquilibre donne gratuitement. Ce chiffre est le plancher à battre.

In [ ]:
# LA RÉFÉRENCE : que vaut un modèle qui répond toujours la même chose ?
# Sans ce chiffre, votre score ne veut rien dire. Notez-le, vous le réutiliserez toute l'année.
reference = 100 * max(y.mean(), 1 - y.mean())

print("la référence, pour tout le semestre \n")
print(f"référence (classe majoritaire) : {reference:.2f} % d'exactitude")

In [ ]:
# À vous — la référence
# Vérifiez la référence à la main : quelle proportion de la cible vaut 1, et que vaut une
# règle qui répond toujours 0 ?

print("proportion d'absents :", round(100*y.mean(), 2), "%")
# TODO : recalculez la référence en une ligne, SANS la variable `reference`,
#        et vérifiez que vous tombez sur le même chiffre.

### Un premier modèle, et une mesure honnête

Un modèle sert à faire mieux que la référence **sur des données qu'il n'a pas vues**. On met donc 20 %
des lignes de côté, il ne les voit jamais, et on mesure sur elles — sinon on mesurerait sa mémoire, pas
sa capacité à décider.
Le modèle employé est un **ensemble d'arbres** : des arbres de décision construits l'un après l'autre,
chacun corrigeant les erreurs du précédent. Il prend les colonnes catégorielles telles quelles, sans
encodage, et traverse les valeurs manquantes sans broncher — c'est ce qui nous épargne du travail ici.

In [ ]:
# un premier modèle, et une mesure honnête sur des données qu'il n'a pas vues
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0)
modele.fit(Xtr, ytr)

score_interne = 100 * modele.score(Xva, yva)

print("le score interne, à comparer à la référence \n")
print(f"score interne : {score_interne:.2f} %   (référence : {reference:.2f} %)")

In [ ]:
# À vous — le premier modèle
# Le score interne vaut mieux que la référence — de combien ?

print("score interne :", round(score_interne, 2), "%   référence :", round(reference, 2), "%")
# TODO : relancez le split avec test_size=0.5. Le score bouge : est-ce le modèle
#        qui a changé, ou la mesure ?

### Les réglages du modèle — ce que vous choisissez, contre ce qu'il apprend

Un modèle manipule deux sortes de nombres. Les **paramètres** sont ce qu'il **apprend** dans les
données : les points de coupe de ses arbres. Personne ne les fixe à la main — ils sortent de
l'entraînement. Les **hyperparamètres** sont ce que **vous** fixez **avant** l'entraînement : le
nombre d'arbres, la taille du pas d'apprentissage, le nombre de feuilles par arbre.
La règle de la cellule qui suit : **un seul hyperparamètre bouge à la fois**, et on mesure. C'est
la discipline du tableau d'essais — une expérience, une variable, une mesure.

In [ ]:
# un hyperparamètre à la fois — et rien d'autre ne bouge
print("le même modèle, sous trois réglages différents \n")
for arbres in (10, 50, 200):
    m = HistGradientBoostingClassifier(max_iter=arbres, random_state=0).fit(Xtr, ytr)
    print(f"  max_iter = {arbres:<4}  {100*m.score(Xva, yva):6.2f} %")

print("→ sous-entraîné (10 arbres), le modèle est en dessous ; au-delà, régler ne rapporte")
print("  plus rien. Le gain n'est pas dans les réglages : il est dans les données.")

In [ ]:
# À vous — les réglages
# Changez UN AUTRE hyperparamètre, seul, et dites ce qu'il change.

# TODO : changez `learning_rate` (0,5 par exemple) SANS toucher à autre chose.
#        Notez la ligne d'essai : hypothèse / changement / attendu / mesuré.

### Prédire, et écrire la soumission

Un modèle qui n'écrit rien ne sert à personne. On prédit sur le test de la compétition et on écrit
`submission.csv` : un identifiant et une prédiction 0/1, une ligne par ligne.
Ce fichier est le livrable : c'est lui qu'on téléverse, et c'est lui qui apparaît au classement. Une
soumission déposée vaut mieux qu'un modèle parfait qui n'existe pas.

In [ ]:
# prédire sur le test et écrire le fichier de soumission
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)

print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")
print("les cinq premières lignes \n")
soumission.head()

In [ ]:
# À vous — la soumission
# Vérifiez que le fichier correspond exactement aux identifiants attendus.

print(soumission.head().to_string(index=False))
# TODO : comptez combien de lignes sont prédites 1. Rapportez ce nombre à la part
#        d'absents de la table : le modèle en prédit-il trop, ou trop peu ?

### Le tableau d'essais — ce qui est lu, ce n'est pas le score

Ce qu'on lit à la fin, ce n'est pas le score : c'est le tableau. Chaque essai porte une hypothèse, un
changement, ce que vous **attendiez**, et ce que vous avez **mesuré**.
La colonne `attendu` ne demande pas de deviner un chiffre — c'est impossible — mais un **signe et une
raison** : mieux que la référence, pire qu'avant, ou pareil, et pourquoi. C'est ce qui transforme
« exécuter puis regarder » en « prédire puis vérifier ».
Un essai qui n'améliore rien est un résultat : il dit où ne pas chercher.

In [ ]:
# le tableau d'essais — c'est lui qui est lu, pas le score
ESSAIS = []

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau des essais. `attendu` : un signe et une raison, jamais une valeur."""
    ESSAIS.append({"hypothèse": hypothese, "changement": changement,
                   "attendu": attendu, "mesuré": round(score, 2)})
    print(pd.DataFrame(ESSAIS).to_string(index=False))

print("la référence, puis un premier essai \n")
noter("rien faire du tout", "classe majoritaire", "c'est le plancher", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

In [ ]:
# À vous — le tableau d'essais
# C'est le livrable de la séance. Ajoutez vos propres lignes — une par essai, avec un
# signe ET une raison dans `attendu`.

# TODO : ajoutez un essai, avec ce que vous attendiez AVANT de voir le résultat.
# noter("votre hypothèse", "ce que vous avez changé", "mieux/pire, et pourquoi", score)

### Explorer la table

Un modèle ne vaut pas mieux que ses données. Avant de toucher au modèle, on lit la table : sa forme,
ses types, ses trous, ses valeurs distinctes, la distribution de la cible. C'est cette lecture qui
produit les **constats** de la séance.
**La règle : un constat sans chiffre n'est pas un constat.** « Il y a des valeurs manquantes » ne dit
rien ; « 5 % sur cette colonne » dit quoi faire.
On explore la table **telle qu'elle est arrivée**, pas la table de travail de la section 1 : ce qu'on a
écarté, il faut d'abord savoir pourquoi — et certaines colonnes sont là pour être trouvées.

### Les trois gestes de l'exploration

- `.info()` et `.head()` : la **forme** — combien de lignes, de colonnes, quels types — et à quoi
  ressemble une ligne. On ne chiffre rien avant d'avoir regardé.
- `.describe()` : **bornes et quartiles** de chaque colonne numérique ; c'est ce qui fait surgir une
  valeur impossible.
- les **graphiques** : une distribution, un taux par groupe. Ils ne remplacent pas les chiffres, ils
  les rendent visibles — et **un graphique sans chiffre ne déclenche aucune décision**.

In [ ]:
# l'inventaire : une ligne par colonne. C'est la feuille de travail de la séance
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

inventaire = pd.DataFrame({
    "type_pandas": [str(t) for t in train.dtypes],
    "non_nuls": train.notna().sum().values,
    "manquants_%": (100 * train.isna().mean()).round(2).values,
    "valeurs_distinctes": [train[c].nunique(dropna=True) for c in train.columns],
    "exemple": [train[c].dropna().iloc[0] if train[c].notna().any() else "" for c in train.columns],
}, index=train.columns)

print("l'inventaire, trié par nombre de valeurs distinctes \n")
print(inventaire.sort_values("valeurs_distinctes").to_string())

In [ ]:
# À vous — l'inventaire — à vous
# L'inventaire a une ligne par colonne. Lisez-le et tirez-en un premier constat.

# TODO : repérez la colonne qui n'a qu'UNE valeur distincte. À quoi sert-elle ?
#        Écrivez le constat, avec un chiffre.

In [ ]:
# la forme de la table, telle qu'elle est arrivée
print("dimensions :", train.shape, "(lignes, colonnes)")
print()
train.info()

In [ ]:
# à quoi ressemble une ligne
print("les cinq premières lignes \n")
print(train.head().to_string(index=False))

print("un tirage au hasard \n")
print(train.sample(3, random_state=0).to_string(index=False))

In [ ]:
# les bornes et les quartiles des colonnes numériques
print("les statistiques des colonnes numériques \n")
print(train.describe().T.round(2).to_string())

In [ ]:
# une distribution, en graphique
colonne = "age"
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(train[colonne].dropna(), bins=40, color="#4a7c9b")
ax.set_title(f"distribution de {colonne}")
ax.set_xlabel(colonne)
ax.set_ylabel("nombre de rendez-vous")
plt.tight_layout()
plt.show()
print("à décrire en une phrase, avec un chiffre.")

In [ ]:
# les types : ce que pandas croit, contre ce que la colonne contient vraiment
print("les colonnes NON numériques, et un exemple de valeur \n")
for c in train.columns:
    if not pd.api.types.is_numeric_dtype(train[c]):
        exemple = train[c].dropna().iloc[0] if train[c].notna().any() else ""
        print(f"  {c:<24} {str(train[c].dtype):<8} exemple : {exemple!r}")

In [ ]:
# les trous, colonne par colonne
taux = (100 * train.isna().mean()).round(2)
taux = taux[taux > 0].sort_values(ascending=False)
print("les colonnes avec des valeurs manquantes (% de la colonne) \n")
print(taux.to_string())
if len(taux):
    fig, ax = plt.subplots(figsize=(7, max(2, 0.3 * len(taux))))
    ax.barh(taux.index[::-1], taux.values[::-1], color="#b5651d")
    ax.set_xlabel("% de valeurs manquantes")
    plt.tight_layout()
    plt.show()

In [ ]:
# le même trou, vu des deux tableaux — c'est ICI qu'on découvre la colonne qui disparaît
print("les manquants : train contre test (écart > 20 points) \n")
for c in train.columns:
    if c not in test.columns:
        continue
    t, e = 100 * train[c].isna().mean(), 100 * test[c].isna().mean()
    if abs(t - e) > 20:
        print(f"  {c:<24} train {t:5.1f} % vide   ·   test {e:5.1f} % vide")

In [ ]:
# la cardinalité : une ligne par colonne, triée par nombre de valeurs distinctes
distinctes = pd.Series({c: train[c].nunique() for c in train.columns}).sort_values(ascending=False)
print("valeurs distinctes par colonne \n")
print(distinctes.to_string())

In [ ]:
# la cible, et où elle se concentre
print("la distribution de la cible \n")
print(f"  {100*y.mean():.2f} % d'absents   ({int(y.sum())} sur {len(y)})")

groupe = "assurance"
taux = (100 * train.groupby(groupe, observed=True)[CIBLE].mean()).sort_values(ascending=False)
print(f"\ntaux d'absence par {groupe} \n")
print(taux.round(2).to_string())

In [ ]:
# les colonnes numériques disent-elles la même chose ?
num = train.select_dtypes("number")
corr = num.corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("corrélation entre les colonnes numériques")
plt.tight_layout()
plt.show()
print("les paires les plus liées \n")
paires = [(corr.columns[i], corr.columns[j], corr.iloc[i, j])
          for i in range(len(corr)) for j in range(i + 1, len(corr))]
for a, b, v in sorted(paires, key=lambda x: -abs(x[2]))[:5]:
    print(f"  {a:<18} {b:<18} {v:+.2f}")

In [ ]:
# les doublons, et les collisions d'identifiants
print("les doublons \n")
print("lignes dupliquées (toutes colonnes confondues) :", int(train.duplicated().sum()))
print("identifiants dupliqués :", int(train[ID].duplicated().sum()))

for col in ("dossier_id", "patient_id"):
    if col in train.columns:
        print(f"  {col:<14} distincts : {train[col].nunique():>7}  ·  lignes : {len(train):>7}")

# Un patient peut revenir plusieurs fois. Combien, au plus ?
if "patient_id" in train.columns:
    print("rendez-vous par patient — max :", int(train["patient_id"].value_counts().max()),
          "· médiane :", int(train["patient_id"].value_counts().median()))

In [ ]:
# À vous — doubler la vérification — à vous
# Combien de lignes sont strictement dupliquées ? Combien de patients reviennent ?

# TODO : trouvez le patient qui a le plus de rendez-vous, et son taux d'absence.
#        Comparez-le au taux global : que raconte ce patient ?

#### Corriger ce qu'on a trouvé — les lignes strictement dupliquées

Une ligne recopiée à l'identique n'est pas un cas limite : c'est un **défaut de collecte**. On la
retire, et on note combien — le nombre fait partie du constat.

In [ ]:
# retirer les lignes strictement dupliquées
avant = len(train)
train_sans_doublons = train.drop_duplicates()

print("les doublons retirés \n")
print("  avant :", avant, "lignes")
print("  après :", len(train_sans_doublons), "lignes")
print("  retirées :", avant - len(train_sans_doublons))

print("→ les retirer est une correction ; mesurer ce que ça change au score est un ARBITRAGE.")

#### Et la référence, quand ne vaut-elle plus rien ?

La classe majoritaire n'est un plancher utile **que dans un cas précis** : classification binaire,
déséquilibrée, jugée à l'exactitude — notre cas.
Sur des classes **équilibrées**, répondre la majorité vaut 50 %, autant que le hasard : le plancher ne
dit rien. Sur une erreur moyenne (régression), le plancher est « toujours la moyenne ». Et sur une
décision par **classement** (les 100 dossiers les plus risqués), il n'y a plus de plancher qui tienne :
on se compare à une règle métier.

In [ ]:
# À vous — l'exploration — à vous
# Un constat sans chiffre n'est pas un constat. Écrivez-en cinq.

# CONSTATS = []
# def constat(observation, chiffre, comparaison, consequence): ...
# TODO : observation · chiffre · comparaison (« par rapport à quoi ? ») · conséquence.

### Un nombre rangé en texte

Le dégât n'est pas du bruit, c'est une **perte silencieuse**. Un modèle d'arbres coupe une colonne en
posant une question du type « cette valeur dépasse-t-elle ce seuil ? ». Pour couper, il faut pouvoir
**comparer** : une colonne rangée en texte est lue comme une catégorie, ses valeurs deviennent des
étiquettes sans ordre, et le modèle perd toute capacité à généraliser sur la grandeur.
Pire : au-delà d'un trop grand nombre de valeurs distinctes, une telle colonne est écartée
automatiquement — l'information disparaît **sans erreur et sans message**.

In [ ]:
# À vous — le nombre rangé en texte — à vous
# Trouvez la colonne de taille et regardez son type.

# TODO : affichez le type de `taille_cm` et sa première valeur. Que faudrait-il
#        faire pour que le modèle puisse la comparer à un seuil ?

#### Corriger ce qu'on a trouvé — la taille rangée en texte

Un nombre rangé en texte n'est pas un arbitrage : c'est une **correction**. On sait ce que la
colonne veut dire — il manque juste le bon type. On la convertit, et on compte ce qui résiste :
une valeur qui ne se convertit pas est un défaut de plus, pas une raison d'abandonner.

In [ ]:
# convertir, et compter ce qui résiste
taille = pd.to_numeric(train["taille_cm"].str.replace(",", "."), errors="coerce")

print("la colonne de taille, après conversion \n")
print("  valeurs converties :", int(taille.notna().sum()))
print("  valeurs non convertibles :", int(taille.isna().sum()))
print("  type obtenu :", taille.dtype)

print("→ le geste est le même pour toute colonne du même genre. La mettre dans la table de")
print("  travail, en revanche, est un ARBITRAGE : cela se mesure, et c'est la séance suivante.")

### La régression logistique, et la table qu'elle exige

Un modèle d'arbres **coupe** ; un modèle **linéaire** calcule : une somme pondérée de ses colonnes,
comprimée en une **probabilité entre 0 et 1**, puis tranchée à **0,5**. Les poids sont ce qu'il apprend.
Cela lui impose deux exigences que l'arbre n'avait pas : il lui faut **des nombres** (une modalité n'a
pas de valeur numérique — il faut une indicatrice 0/1 par modalité) et **pas de trous** (un manquant
n'a pas de case dans la somme — il faut le combler).

In [ ]:
# la régression logistique, et la table qu'il exige.
# Un modèle linéaire ne lit pas la table telle quelle, ni avec des trous : on la lui prépare.
from sklearn.linear_model import LogisticRegression

# 1. Où sont les trous ? On les compte avant de les traiter.
manquants = X.isna().sum()
manquants = manquants[manquants > 0].sort_values(ascending=False)

print("colonnes avec des valeurs manquantes (nombre) \n")
print(manquants.to_string())

# 2. Combler les trous : chaque trou reçoit la MÉDIANE de sa colonne — l'imputation minimale.
#    On comble plutôt que de supprimer : supprimer des lignes, c'est perdre des rendez-vous.
X_num = X.fillna(X.median(numeric_only=True))

# 3. Les catégorielles deviennent des indicatrices : une colonne 0/1 par modalité (« one-hot »).
#    Huit colonnes de texte se transforment en des dizaines de colonnes de 0/1.
X_num = pd.get_dummies(X_num, columns=CAT)

print("après préparation \n")
print("colonnes :", X.shape[1], "->", X_num.shape[1])
print("colonnes non numériques restantes :",
      sum(1 for c in X_num.columns if not pd.api.types.is_numeric_dtype(X_num[c])))

# Entraîner et mesurer, comme pour les arbres. « Solveur » = la méthode que le modèle emploie
#    pour trouver ses poids. Celui employé ici est adapté à un tableau long et étroit : il converge
#    en quelques secondes là où le solveur par défaut traîne et affiche un avertissement. On
#    l'emploie, on ne le règle pas : le réglage n'est pas au programme d'aujourd'hui.
Xtr_n, Xva_n, ytr_n, yva_n = train_test_split(X_num, y, test_size=0.2, stratify=y, random_state=0)

modele_lineaire = LogisticRegression(solver="newton-cholesky")
modele_lineaire.fit(Xtr_n, ytr_n)

score_lineaire = 100 * modele_lineaire.score(Xva_n, yva_n)

print("le score du linéaire, à comparer aux arbres \n")
print(f"linéaire : {score_lineaire:.2f} %   (arbres : {score_interne:.2f} %)")

noter("linéaire contre arbres", "LogisticRegression",
      "mieux que la référence", score_lineaire)

# 5. Comment il décide : il calcule une PROBABILITÉ par ligne, puis tranche à 0,50.
#    `predict_proba` montre cette probabilité ; `predict` — déjà appelé plus haut — applique le seuil.
probas = modele_lineaire.predict_proba(Xva_n)[:, 1]

print("les probabilités calculées sur la validation \n")
print(pd.Series(probas).describe().round(3).to_string())

print("au-dessus du seuil de 0,50 \n")
print(int((probas >= 0.50).sum()), "lignes sur", len(probas), "que le modèle déclare absentes")

In [ ]:
# À vous — le modèle linéaire — à vous
# Le linéaire est en dessous des arbres ici — et il a fallu lui préparer la table nous-mêmes.

print("linéaire :", round(score_lineaire, 2), "%   arbres :", round(score_interne, 2), "%")
# TODO : dites en deux lignes ce que le linéaire coûte en préparation,
#        et ce qu'il apporte en échange.

### Une feature de votre cru — une seule, des deux côtés

Une feature n'est pas une idée : c'est une **hypothèse**. Elle dit ce qu'elle raconte, et on la mesure
**seule** — dix features ajoutées d'un coup, et vous ne saurez pas ce qui a agi.
Elle doit s'appliquer **des deux côtés**, entraînement et test : un modèle qui apprend sur une version
des données et prédit sur une autre commet une erreur que rien ne signale.

In [ ]:
# votre feature — une seule, appliquée des deux côtés, sinon décalage
# silencieux : le modèle apprend sur une version des données et prédit sur une autre.
# Votre fonction doit fabriquer la colonne à partir des colonnes d'origine, et donner le même
# résultat chaque fois qu'on l'appelle : vous relancerez souvent cette cellule.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre, et dites en une phrase ce qu'elle raconte :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Ajoutez votre feature aux DEUX jeux, PUIS relancez « un premier modèle »
# et « prédire et écrire la soumission » :
# X  = ajouter_features(X)
# Xt = ajouter_features(Xt)
# noter("une feature de plus", "votre feature",
#       "un signe et une raison", score_interne)

In [ ]:
# À vous — votre feature — à vous
# Écrivez la vôtre — une seule — et appliquez-la des deux côtés.

# def ajouter_features(df):
#     df = df.copy()
#     df["..."] = ...          # dites en une phrase ce qu'elle raconte
#     return df
# TODO : écrivez-la ci-dessus, appliquez-la à X ET à Xt, puis relancez la section 3.

### L'interface — un modèle qu'on peut appeler

Votre enseignant ouvrira votre notebook, ira à la cellule `predict`, et l'exécutera sur le tableau de
test — **sans lire le reste**. C'est la raison d'être de cette fonction : elle prend un tableau brut et
rend un 0/1 par ligne, en refaisant *toutes* les opérations de la chaîne (colonnes écartées, modalités,
modèle).
Écrire cette fonction est aussi le moment où vous êtes obligé de décider ce qui passe par la chaîne —
la règle des deux côtés, rendue exécutable. Le test qui suit vérifie qu'elle répond ce qu'on attend.

In [ ]:
# l'interface imposée, et son test
def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours
    appellera — pas le notebook. Gardez-la vivante en même temps que vous essayez vos idées."""
    d = df.drop(columns=[c for c in A_ECARTER + TROP_FIN if c in df.columns], errors="ignore")

    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)

    # L'interface porte la MÊME transformation que l'entraînement — votre feature comprise.
    d = ajouter_features(d)

    return modele.predict(d).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"

    entiers = sum(p)

    assert 0 <= entiers <= 500, "nombre de 1 incohérent"
    print(f"test OK — 500 prédictions, {entiers} positifs")

test_predict()

In [ ]:
# À vous — l'interface — à vous
# Vérifiez que l'interface répond, et ce qu'elle fait de plus que `modele.predict`.

print("prédictions sur 500 lignes :", predict(test.head(500)).shape)
# TODO : cassez volontairement l'interface (écartez une colonne dans `predict`)
#        et regardez le test échouer. Qu'est-ce qui vous l'a signalé ?

### La cible est un objet suspect

Une cible n'est pas une vérité tombée du ciel : **quelqu'un l'a produite**, à un moment, avec un
protocole. Tant qu'on ne pose pas la question, on entraîne un modèle à reproduire le travail de
quelqu'un d'autre — erreurs comprises.
**En fraude, c'est le cas d'école** : l'étiquette « fraude » vient d'un impayé, qui arrive trente
à quatre-vingt-dix jours après la transaction. Les lignes récentes sont donc étiquetées
« honnête » — non parce qu'elles le sont, mais parce que l'impayé n'est pas encore arrivé. Le
modèle apprend « récent = honnête », et rate la fraude du mois.
**Deux guichetiers, deux définitions** : l'un coche « absent » pour un rendez-vous annulé,
l'autre non. Une partie des lignes est étiquetée par **habitude** — et le modèle apprend le
désaccord.
La question n'est donc pas « la cible est-elle juste ? » mais **« qui l'a produite, quand, et
avec quelle règle ? »**.

In [ ]:
# À vous — la cible
# Regardez `motif_absence` : elle n'est renseignée que pour certaines lignes, jamais pour toutes.

# TODO : combien de lignes portent un motif d'absence ? Et combien d'absents n'en portent aucun ?
#        Que vous dit cet écart sur la façon dont cette colonne a été remplie ?

### Les valeurs aberrantes — « est-ce possible ? »

Un solde négatif est légitime : un compte peut être débiteur. Un âge négatif ne l'est pas. La question
n'est donc jamais « est-ce loin de la moyenne ? » mais **« est-ce possible ? »**.
On regarde les bornes de chaque colonne et on compte les valeurs impossibles. Supprimer des extrêmes
sans justification est l'erreur la plus fréquente des débutants — en détection de fraude, l'extrême est
précisément le signal.

In [ ]:
# les valeurs aberrantes : erreur de saisie ou cas extrême LÉGITIME ?
# On ne supprime rien sans justification chiffrée. On regarde d'abord ce qui est possible.
print("les bornes de chaque colonne \n")
for col in ("age", "solde_credit", "montant_acte", "delai_jours", "taille_cm"):
    if col not in train.columns:
        continue
    s = train[col]
    if pd.api.types.is_numeric_dtype(s):
        print(f"  {col:<16} min {s.min():>10.2f} · max {s.max():>10.2f} · négatifs {int((s < 0).sum())}")
    else:
        print(f"  {col:<16} type texte · {s.nunique()} modalités distinctes")

print("\n→ un maximum très grand peut être un vrai cas extrême. Un minimum impossible est un bug.")
print("  La question n'est pas « est-ce loin de la moyenne ? » mais « est-ce possible ? »")

In [ ]:
# À vous — les valeurs aberrantes — à vous
# Une valeur impossible est-elle présente ? Laquelle, et combien de lignes ?

# TODO : cherchez une colonne dont le minimum est impossible (indice : l'âge).
#        Comptez les lignes concernées, et dites ce qu'il faut en faire — avec un chiffre.

#### Corriger ce qu'on a trouvé — une valeur impossible

Un âge de −1 n'est pas un cas extrême : c'est un **enregistrement cassé**. On ne le « corrige »
pas — on ne connaît pas le vrai âge — on le **marque comme manquant**, pour que rien ne s'en
serve comme d'un nombre. Et on vérifie au passage si la casse laisse d'autres traces.

In [ ]:
# marquer, pas corriger
impossible = train["age"] == -1

print("les âges impossibles \n")
print("  lignes concernées :", int(impossible.sum()))
print("  leur date de naissance est aussi absente :",
      bool(train.loc[impossible, "date_naissance"].eq("").all()))

print("→ un enregistrement cassé l'est souvent à plusieurs endroits : la date de naissance est")
print("  vide sur les mêmes lignes. On marque, et on n'invente pas.")

### La colonne qui disparaît en production

L'exploration a montré une colonne **pleine dans l'historique et vide dans le test** : le système qui
l'alimentait a cessé de le faire. Le modèle apprend alors avec une information qu'il n'aura plus le jour
où il sert — et rien ne l'avertit.
Ce que ça coûte, on ne le suppose pas : **on le mesure**.

In [ ]:
# ce que coûte la colonne qui disparaît : on la remet, on mesure, on la retire, on remesure
def mesurer(avec_colonne):
    Xa = X.copy()
    if avec_colonne:
        Xa["canal_prise"] = train.loc[Xa.index, "canal_prise"].fillna("").astype("category")
    Xtr_m, Xva_m, ytr_m, yva_m = train_test_split(Xa, y, test_size=0.2, stratify=y, random_state=0)
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_m, ytr_m).score(Xva_m, yva_m)

avec, sans = mesurer(True), mesurer(False)
print("la colonne réintroduite, mesurée sur la validation \n")
print(f"  avec : {avec:.2f} %   ·   sans : {sans:.2f} %   ·   ce qu'elle apporte : {avec - sans:+.2f} pt")
print()
print("au classement, elle est vide pour tout le monde : ce gain-là ne sera pas rendu.")

In [ ]:
# À vous — la colonne qui disparaît — à vous
# Le gain interne est réel. Et au classement ?

# TODO : dites en une phrase pourquoi un gain mesuré ici ne se retrouvera pas
#        sur le test. Notez-le comme essai : "mieux en interne, et alors ?".

### Les colonnes connues après l'issue

Certaines colonnes ne sont renseignées qu'**après** le rendez-vous : la durée de la consultation, le
motif d'absence. Elles « savent » la réponse.
On le démontre en les réintroduisant dans la table — et c'est ici qu'arrive la phrase à retenir :
**un score de 0,99 obtenu en cinq minutes est toujours un bug, jamais un talent.** Ce n'est pas le
modèle qui est bon, c'est la donnée qui portait la réponse.

In [ ]:
# une colonne peut-elle renseigner une information connue APRÈS l'issue ?
# Une colonne remplie APRÈS le rendez-vous : elle n'existe pas encore au moment où l'on décide.
# Méthode : pour chaque colonne, se demander « cette valeur est-elle connue le jour de la prise
# du rendez-vous ? ». Puis vérifier par la mesure : sa présence seule suffit-elle à prédire ?
soupcons = []
for c in train.columns:
    if c in (ID, CIBLE) or not pd.api.types.is_numeric_dtype(train[c]):
        continue
    # une colonne qui ne prédit QUE les absents, par construction, est suspecte
    if train[c].isna().any():
        taux_manquant_absents = train.loc[train[CIBLE] == 1, c].isna().mean()
        taux_manquant_presents = train.loc[train[CIBLE] == 0, c].isna().mean()
        if abs(taux_manquant_absents - taux_manquant_presents) > 0.5:
            soupcons.append((c, round(100 * taux_manquant_absents, 1),
                             round(100 * taux_manquant_presents, 1)))

print("les colonnes suspectes, par leurs manquants \n")
if soupcons:
    print(pd.DataFrame(soupcons, columns=["colonne", "% vide chez les absents",
                                          "% vide chez les présents"]).to_string(index=False))
else:
    print("aucune colonne ne se distingue par ses manquants — cherchez par le TEMPS, pas par le vide")

In [ ]:
# une colonne remplie APRÈS le rendez-vous, réintroduite dans la table
X_post = X.copy()
X_post["duree_consultation"] = train.loc[X_post.index, "duree_consultation"]
Xtr_p, Xva_p, ytr_p, yva_p = train_test_split(X_post, y, test_size=0.2, stratify=y, random_state=0)
score_post = 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_p, ytr_p).score(Xva_p, yva_p)

print("le même modèle, avec une colonne qui n'existe qu'après le rendez-vous \n")
print(f"  sans : {score_interne:.2f} %   ·   avec : {score_post:.2f} %")
print()
print("un score de 0,99 obtenu en cinq minutes est toujours un bug, jamais un talent.")

In [ ]:
# À vous — les colonnes connues après l'issue — à vous
# Pourquoi une durée de consultation prédit-elle l'absence presque parfaitement ?

# TODO : dites, en une phrase et sans regarder le code, pourquoi cette colonne
#        « connaît la réponse ». Est-elle renseignée pour tout le monde ?

### Ce qu'on rend — le rapport

Ce qu'on rend n'est pas ce notebook : c'est un **texte**. Cinq constats chiffrés — observation,
chiffre, comparaison, conséquence — et trois hypothèses de features.
Un constat a quatre morceaux, et le troisième est celui qui manque toujours : **par rapport à quoi ?**
Sans comparaison, un chiffre n'est pas un constat. Une hypothèse, elle, est une **prédiction
falsifiable** : ce qu'elle raconte, comment on la teste, et ce qui la contredirait.

In [ ]:
# le gabarit. Remplissez-le, cinq fois. Un constat sans chiffre sera rendu sans note
CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (une autre colonne, l'ensemble, un seuil)
       conséquence : ce qu'on en fait — garder, écarter, imputer, surveiller"""
    CONSTATS.append({"observation": observation, "chiffre": chiffre,
                     "comparaison": comparaison, "conséquence": consequence})
    print(pd.DataFrame(CONSTATS).to_string(index=False))

# Trois exemples — à remplacer par les vôtres :
print("un exemple de constat \n")
constat("une colonne ne sert à rien", "AUC 0,501", "1 valeur distincte sur les 80 cliniques",
        "écarter")

print("un deuxième exemple \n")
constat("deux colonnes disent la même chose", "corrélation 0,99", "contre 0,3 en moyenne ailleurs",
        "n'en garder qu'une")

print("un troisième exemple \n")
constat("la cible n'est pas équilibrée", f"{100*y.mean():.1f} % de positifs",
        "contre 50 % pour un problème équilibré", "le dire, avec le chiffre")

In [ ]:
# le gabarit des hypothèses
HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction falsifiable : ce qu'elle raconte, comment on la teste,
       et ce qui la contredirait."""
    HYPOTHESES.append({"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
                       "test": comment_on_la_test, "falsifiée si": ce_qui_la_contredirait})
    print(pd.DataFrame(HYPOTHESES).to_string(index=False))

# Trois exemples — à remplacer :
print("un exemple d'hypothèse \n")
hypothese("jours depuis la prise", "un délai long fait oublier le rendez-vous",
          "score avec / sans, à modèle constant", "écart < 0,2 pt")

print("une deuxième hypothèse \n")
hypothese("nombre de rendez-vous passés du patient", "un habitué manque moins",
          "score avec / sans", "écart nul sur le test")

print("une troisième hypothèse \n")
hypothese("jour de la semaine", "le lundi n'est pas le samedi",
          "taux d'absence par jour, seaux ≥ 500 lignes", "écart < 1 pt")

In [ ]:
# et ce qu'on rend : le rapport
# Le rapport n'est pas un notebook : c'est un TEXTE, avec cinq lignes de constats et trois
# hypothèses. Le voici en clair, prêt à copier dans votre dépôt.
print("CONSTATS CHIFFRÉS \n")
print(pd.DataFrame(CONSTATS).to_string(index=False))

print("HYPOTHÈSES DE FEATURES \n")
print(pd.DataFrame(HYPOTHESES).to_string(index=False))

print("Un constat sans chiffre n'est pas un constat. Une hypothèse sans falsification n'en est pas une.")

In [ ]:
# À vous — le rapport — à vous
# Cinq constats + trois hypothèses. C'est ça qu'on rend.

# TODO : vérifiez que chaque constat porte une COMPARAISON, et que chaque
#        hypothèse porte ce qui la CONTREDIRAIT. Sans ces morceaux, c'est à refaire.

### Le dépôt

Vos commits portent votre identité : c'est ce qui permet d'attribuer le travail. Cette cellule vérifie
l'**état**, jamais la valeur — inutile d'imprimer votre adresse à l'écran, et un notebook forké emporte
ses sorties.

In [ ]:
# le dépôt (fait en séance, 10 minutes)
#   git init && git add . && git commit -m "J1" && git remote add origin <votre dépôt>
#   git push -u origin main
# Votre identité n'est à renseigner que si vous committez depuis un terminal (local ou
# Codespaces) : par l'interface web, GitHub signe le commit avec votre compte.
import subprocess

try:
    for cle in ("user.name", "user.email"):
        r = subprocess.run(["git", "config", "--global", cle], capture_output=True, text=True)
        v = r.stdout.strip()
        # On affiche l'ÉTAT, pas la valeur : inutile d'imprimer votre adresse à l'écran,
        # et un notebook forké emporte ses sorties avec lui.
        print(f"  git {cle:<10} : {'renseigné' if v else 'NON RENSEIGNÉ — à faire'}")
except FileNotFoundError:
    print("git n'est pas disponible ici — les commandes ci-dessus sont à passer en local")